In [0]:
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown(
    "schema",
    "migration_dev",
    ["migration_dev", "migration_test", "migration_prod_sim"],
)
dbutils.widgets.text("batch_id", "batch_001")

catalog_name = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog_name):
    raise ValueError("Invalid catalog name")

if schema_name not in {
    "migration_dev", "migration_test", "migration_prod_sim"
}:
    raise ValueError("Invalid environment schema")

if not re.fullmatch(r"batch_[0-9]+", batch_id):
    raise ValueError("Invalid batch ID")

table_prefix = f"{catalog_name}.{schema_name}"
volume_root = f"/Volumes/{catalog_name}/{schema_name}"

print(f"Environment: {table_prefix}")
print(f"Batch: {batch_id}")
print(f"Volume root: {volume_root}")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW file_reconciliation_checks AS
WITH metadata AS (
    SELECT *
    FROM IDENTIFIER(:catalog || '.' || :schema || '.bronze_metadata')
    WHERE batch_id = :batch_id
),
inventory AS (
    SELECT *
    FROM IDENTIFIER(:catalog || '.' || :schema || '.bronze_file_inventory')
    WHERE batch_id = :batch_id
),
reconciliation AS (
    SELECT
        COALESCE(m.file_name, i.file_name) AS file_name,
        CASE
            WHEN m.source_file IS NULL
                THEN 'EXTRA_PDF'
            WHEN i.pdf_path IS NULL
                THEN 'MISSING_PDF'
            WHEN m._rescued_data IS NOT NULL
                THEN 'METADATA_PARSE_ERROR'
            WHEN i.file_size_bytes <= 0
                THEN 'EMPTY_PDF'
            WHEN m.pdf_path IS NULL
                OR REGEXP_REPLACE(m.pdf_path, '^dbfs:', '') <> i.pdf_path
                THEN 'PATH_MISMATCH'
            WHEN m.source_sha256 IS NULL
                OR LOWER(TRIM(m.source_sha256)) <> i.actual_sha256
                THEN 'CHECKSUM_MISMATCH'
            ELSE 'MATCHED'
        END AS reconciliation_status
    FROM metadata m
    FULL OUTER JOIN inventory i
        ON m.batch_id = i.batch_id
       AND m.file_name = i.file_name
)
SELECT
    reconciliation_status,
    COUNT(*) AS record_count
FROM reconciliation
GROUP BY reconciliation_status
ORDER BY reconciliation_status;

In [0]:
%sql
SELECT * FROM file_reconciliation_checks;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW metadata_quality_checks AS
WITH metadata AS (
    SELECT *
    FROM IDENTIFIER(:catalog || '.' || :schema || '.bronze_metadata')
    WHERE batch_id = :batch_id
),
checks AS (
    SELECT 'DUPLICATE_DOCUMENT_ID' AS check_name, COUNT(*) AS failures
    FROM (
        SELECT document_id
        FROM metadata
        GROUP BY document_id
        HAVING COUNT(*) > 1
    )

    UNION ALL

    SELECT 'DUPLICATE_FILE_NAME', COUNT(*)
    FROM (
        SELECT file_name
        FROM metadata
        GROUP BY file_name
        HAVING COUNT(*) > 1
    )

    UNION ALL

    SELECT 'MISSING_REQUIRED_FIELDS', COUNT(*)
    FROM metadata
    WHERE EXISTS(
        ARRAY(
            document_id, account_number, customer_name,
            document_type, statement_date, file_name,
            pdf_path, source_sha256
        ),
        value -> value IS NULL OR TRIM(value) = ''
    )

    UNION ALL

    SELECT 'INVALID_STATEMENT_DATE', COUNT(*)
    FROM metadata
    WHERE TRY_CAST(statement_date AS DATE) IS NULL

    UNION ALL

    SELECT 'INVALID_CHECKSUM_FORMAT', COUNT(*)
    FROM metadata
    WHERE source_sha256 IS NULL
       OR NOT (source_sha256 RLIKE '^[0-9a-fA-F]{64}$')

    UNION ALL

    SELECT 'UNSUPPORTED_DOCUMENT_TYPE', COUNT(*)
    FROM metadata
    WHERE document_type IS NULL
       OR document_type <> 'ACCOUNT_STATEMENT'
)
SELECT
    check_name,
    failures,
    CASE WHEN failures = 0 THEN 'PASS' ELSE 'FAIL' END AS result
FROM checks
ORDER BY check_name;

In [0]:
%python
import json
from pathlib import Path
from pyspark.sql import functions as F

Path(f"{volume_root}/landing/{batch_id}/manifest.json")

manifest_path = Path(
    f"/Volumes/workspace/migration_dev/landing/{batch_id}/manifest.json"
)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

metadata_count = (
    spark.table(f"{table_prefix}.bronze_metadata")
    .filter(F.col("batch_id") == batch_id)
    .count()
)

pdf_count = (
    spark.table(f"{table_prefix}.bronze_file_inventory")
    .filter(F.col("batch_id") == batch_id)
    .count()
)

expected_metadata = manifest["expected_metadata_count"]
expected_pdfs = manifest["expected_document_count"]

print(f"Metadata: expected={expected_metadata}, actual={metadata_count}")
print(f"PDFs: expected={expected_pdfs}, actual={pdf_count}")

if metadata_count != expected_metadata or pdf_count != expected_pdfs:
    raise RuntimeError("FAIL: Counts do not match the source manifest")

print("PASS: Both counts match the source manifest")

In [0]:
%sql
SELECT
    CONCAT('FILE_', reconciliation_status) AS check_name,
    CASE
        WHEN reconciliation_status = 'MATCHED' THEN 0
        ELSE record_count
    END AS failures
FROM file_reconciliation_checks

UNION ALL

SELECT check_name, failures
FROM metadata_quality_checks

ORDER BY check_name;

In [0]:
%python
import json
import uuid
from pathlib import Path
from pyspark.sql import functions as F

Path(f"{volume_root}/landing/{batch_id}/manifest.json")
validation_run_id = str(uuid.uuid4())

manifest = json.loads(
    Path(
        f"/Volumes/workspace/migration_dev/landing/{batch_id}/manifest.json"
    ).read_text(encoding="utf-8")
)

# Read the reusable validation checks.
quality_checks = spark.sql("""
    SELECT
        CONCAT('FILE_', reconciliation_status) AS check_name,
        CAST(
            CASE WHEN reconciliation_status = 'MATCHED'
                 THEN 0 ELSE record_count END
            AS BIGINT
        ) AS failures
    FROM file_reconciliation_checks

    UNION ALL

    SELECT check_name, CAST(failures AS BIGINT)
    FROM metadata_quality_checks
""")

# Recalculate counts rather than reuse earlier notebook values.
actual_metadata = (
    spark.table(f"{table_prefix}.bronze_metadata")
    .filter(F.col("batch_id") == batch_id)
    .count()
)

actual_documents = (
    spark.table(f"{table_prefix}.bronze_file_inventory")
    .filter(F.col("batch_id") == batch_id)
    .count()
)

count_checks = spark.createDataFrame(
    [
        (
            "METADATA_COUNT_MISMATCH",
            int(actual_metadata != manifest["expected_metadata_count"]),
        ),
        (
            "PDF_COUNT_MISMATCH",
            int(actual_documents != manifest["expected_document_count"]),
        ),
    ],
    "check_name STRING, failures BIGINT",
)

results = (
    quality_checks.unionByName(count_checks)
    .withColumn("validation_run_id", F.lit(validation_run_id))
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn(
        "result",
        F.when(F.col("failures") == 0, "PASS").otherwise("FAIL"),
    )
    .withColumn("checked_at", F.current_timestamp())
    .select(
        "validation_run_id", "batch_id", "check_name",
        "failures", "result", "checked_at",
    )
)

(
    results.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{table_prefix}.validation_results")
)

saved_results = (
    spark.table(f"{table_prefix}.validation_results")
    .filter(F.col("validation_run_id") == validation_run_id)
)

print(f"Validation run ID: {validation_run_id}")
display(saved_results.orderBy("check_name"))

In [0]:
%python
from pyspark.sql import functions as F

# Read only the results from this validation run.
checks = (
    spark.table(f"{table_prefix}.validation_results")
    .filter(
        (F.col("validation_run_id") == validation_run_id)
        & (F.col("batch_id") == batch_id)
    )
    .collect()
)

required_checks = {
    "FILE_MATCHED",
    "DUPLICATE_DOCUMENT_ID",
    "DUPLICATE_FILE_NAME",
    "MISSING_REQUIRED_FIELDS",
    "INVALID_STATEMENT_DATE",
    "INVALID_CHECKSUM_FORMAT",
    "UNSUPPORTED_DOCUMENT_TYPE",
    "METADATA_COUNT_MISMATCH",
    "PDF_COUNT_MISMATCH",
}

recorded_names = [row["check_name"] for row in checks]
missing_checks = required_checks - set(recorded_names)
duplicate_checks = len(recorded_names) - len(set(recorded_names))

failed_checks = sum(
    1 for row in checks
    if row["result"] != "PASS" or row["failures"] != 0
)
failed_checks += len(missing_checks) + duplicate_checks

# A source batch must also be declared ready.
if manifest.get("status") != "READY" or manifest.get("batch_id") != batch_id:
    failed_checks += 1

batch_status = "APPROVED" if failed_checks == 0 else "REJECTED"

decision = spark.createDataFrame(
    [(
        batch_id,
        validation_run_id,
        batch_status,
        manifest["expected_document_count"],
        actual_documents,
        manifest["expected_metadata_count"],
        actual_metadata,
        failed_checks,
    )],
    """
    batch_id STRING,
    validation_run_id STRING,
    batch_status STRING,
    expected_document_count BIGINT,
    actual_document_count BIGINT,
    expected_metadata_count BIGINT,
    actual_metadata_count BIGINT,
    failed_checks BIGINT
    """,
).withColumn("updated_at", F.current_timestamp())

decision.createOrReplaceTempView("current_batch_decision")

# Update the existing batch row, or insert it on the first run.
spark.sql(f"""
    MERGE INTO {table_prefix}.batch_control AS target
    USING current_batch_decision AS source
    ON target.batch_id = source.batch_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

display(
    spark.table(f"{table_prefix}.batch_control")
    .filter(F.col("batch_id") == batch_id)
)

if batch_status == "REJECTED":
    raise RuntimeError(
        f"Batch rejected. Failed or incomplete checks: {failed_checks}; "
        f"missing checks: {sorted(missing_checks)}"
    )

print(f"{batch_id}: APPROVED")